# 🏆 Hull - HYBRID Architecture: Complex OOF + Simple Inference

## 🎯 Architecture Philosophy

**Dual-Model Approach:**

**PARTIE A - Complex Model (Offline):**
- Full feature engineering (rolling, regime, interactions)
- IN-FOLD feature selection (stability + clustering)
- Expanding window CV (5 folds)
- GridSearch LightGBM (125 combos per fold)
- Base models (3x ensemble) + Meta model (stacking)
- **PURPOSE**: Generate high-quality OOF predictions

**PARTIE B - Multiplier Optimization:**
- Use OOF meta predictions from complex model
- Optimize multiplier (return → position mapping)
- Maximize competition score

**PARTIE C - Simple Inference Model:**
- RAW features ONLY (no rolling/regime)
- Single LightGBM model
- Uses optimized multiplier from Part B
- **PURPOSE**: Kaggle row-by-row inference

**PARTIE D - Kaggle predict() function:**
- Autonomous, standalone
- Uses simple model + multiplier
- Clip output to [0, 2]

---

In [33]:
# ===================================================================
# IMPORTS
# ===================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Tuple, Dict
from itertools import product
import time
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import AgglomerativeClustering
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (16, 6)
print('✅ Imports successful')
print(f'LightGBM version: {lgb.__version__}')


✅ Imports successful
LightGBM version: 4.5.0


## 📊 Load Data

In [34]:
print('Loading data...')
train = pd.read_csv('train.csv')
test = pd.read_csv("test.csv")
train = train.sort_values('date_id').reset_index(drop=True)
test = test.sort_values('date_id').reset_index(drop=True)
print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')
print(f'Date range:  {train["date_id"].min()} to {train["date_id"].max()}')
feature_prefixes = ['D', 'E', 'I', 'M', 'P', 'S', 'V']
raw_features = [col for col in train.columns if any(col.startswith(p) for p in feature_prefixes)]
target_col = 'market_forward_excess_returns'
print(f'\nRaw features: {len(raw_features)}')
print(f'Target: {target_col}')


Loading data...
Train shape: (9048, 98)
Test shape:  (10, 99)
Date range:  0 to 9047

Raw features: 94
Target: market_forward_excess_returns


## 🔧 Competition Score Function

In [35]:
def competition_score(strategy_returns, risk_free_returns, market_returns):
    """
    Hull competition metric implementation.
    
    Returns: float score (higher is better)
    """
    strategy_excess = strategy_returns - risk_free_returns
    market_excess = market_returns - risk_free_returns
    # Geometric mean
    strat_cum = (1 + strategy_excess).prod()
    strat_mean = strat_cum ** (1 / len(strategy_returns)) - 1
    market_cum = (1 + market_excess).prod()
    market_mean = market_cum ** (1 / len(market_returns)) - 1
    # Volatility
    strat_std = strategy_returns.std()
    market_std = market_returns.std()
    
    if strat_std == 0 or market_std == 0:
        return -999
    sharpe = (strat_mean / strat_std) * np.sqrt(252)
    # Penalties
    strat_vol_ann = strat_std * np.sqrt(252) * 100
    mkt_vol_ann = market_std * np.sqrt(252) * 100
    excess_vol = max(0, strat_vol_ann / mkt_vol_ann - 1.2)
    vol_penalty = 1 + excess_vol
    return_gap = max(0, (market_mean - strat_mean) * 100 * 252)
    return_penalty = 1 + (return_gap ** 2) / 100
    return float(sharpe / (vol_penalty * return_penalty))
print('✅ Competition score function defined')


✅ Competition score function defined


---
# PARTIE A: COMPLEX MODEL (Offline - OOF Generation)
---

## A1. Safe Feature Engineering Functions

In [36]:
def create_safe_rolling_features(df: pd.DataFrame, cols: List[str], windows: List[int] = [5, 10, 21]) -> pd.DataFrame:
    """
    Create safe rolling features using ONLY past data.
    """
    df_out = df.copy()

    for col in cols:
        if col not in df.columns:
            continue

        # Lags
        df_out[f'{col}_lag1'] = df[col].shift(1)
        df_out[f'{col}_lag2'] = df[col].shift(2)
        df_out[f'{col}_lag5'] = df[col].shift(5)

        # Rolling statistics
        for w in windows:
            min_p = max(3, w // 2)
            df_out[f'{col}_mean_{w}'] = df[col].shift(1).rolling(w, min_periods=min_p).mean()
            df_out[f'{col}_std_{w}'] = df[col].shift(1).rolling(w, min_periods=min_p).std()

        # Skewness (21-day only)
        df_out[f'{col}_skew_21'] = df[col].shift(1).rolling(21, min_periods=10).skew()

        # Momentum (5-day)
        df_out[f'{col}_mom_5'] = df[col].shift(1) / (df[col].shift(6) + 1e-8) - 1

    return df_out


def create_safe_regime_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create safe regime features (rolling volatility).
    """
    df_out = df.copy()

    if 'forward_returns' in df.columns:
        df_out['rolling_vol_21'] = df['forward_returns'].shift(1).rolling(21, min_periods=10).std()

    return df_out


print('Feature engineering functions defined')


def create_cross_ratio_features(df: pd.DataFrame, raw_features: List[str]) -> pd.DataFrame:
    """
    PHASE 2: Create cross-ratio features between different feature types.
    Captures relationships like M/V (momentum/volatility), D/E (price/economic), etc.
    """
    df_out = df.copy()

    M_cols = [c for c in raw_features if c.startswith('M')][:5]
    V_cols = [c for c in raw_features if c.startswith('V')][:5]
    D_cols = [c for c in raw_features if c.startswith('D')][:5]
    E_cols = [c for c in raw_features if c.startswith('E')][:5]

    # M/V ratios (momentum/volatility = risk-adjusted momentum)
    for m_col in M_cols:
        for v_col in V_cols:
            if m_col in df.columns and v_col in df.columns:
                df_out[f'{m_col}_div_{v_col}'] = df[m_col] / (np.abs(df[v_col]) + 1e-8)

    # D/E ratios (price/economic indicators)
    for d_col in D_cols:
        for e_col in E_cols:
            if d_col in df.columns and e_col in df.columns:
                df_out[f'{d_col}_div_{e_col}'] = df[d_col] / (np.abs(df[e_col]) + 1e-8)

    # Clean inf/nan
    df_out = df_out.replace([np.inf, -np.inf], np.nan).fillna(0)

    return df_out


print('Cross-ratio feature function defined')


def create_advanced_regime_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    PHASE 3: Create advanced regime features based on market conditions.
    Identifies volatility, trend, and momentum regimes.
    """
    df_out = df.copy()

    if 'forward_returns' not in df.columns:
        return df_out

    # Volatility regime (3 levels: low, medium, high)
    vol_21 = df['forward_returns'].shift(1).rolling(21, min_periods=10).std()
    vol_quantiles = vol_21.quantile([0.33, 0.67])
    df_out['vol_regime'] = 1  # medium
    df_out.loc[vol_21 <= vol_quantiles.iloc[0], 'vol_regime'] = 0  # low
    df_out.loc[vol_21 >= vol_quantiles.iloc[1], 'vol_regime'] = 2  # high

    # Trend regime (up/down/sideways)
    ma_short = df['forward_returns'].shift(1).rolling(5, min_periods=3).mean()
    ma_long = df['forward_returns'].shift(1).rolling(21, min_periods=10).mean()
    trend = ma_short - ma_long
    df_out['trend_regime'] = 0  # sideways
    df_out.loc[trend > 0.0005, 'trend_regime'] = 1  # up
    df_out.loc[trend < -0.0005, 'trend_regime'] = -1  # down

    # Momentum regime (3 levels)
    mom_10 = df['forward_returns'].shift(1).rolling(10, min_periods=5).mean()
    mom_quantiles = mom_10.quantile([0.33, 0.67])
    df_out['momentum_regime'] = 1  # medium
    df_out.loc[mom_10 <= mom_quantiles.iloc[0], 'momentum_regime'] = 0  # low
    df_out.loc[mom_10 >= mom_quantiles.iloc[1], 'momentum_regime'] = 2  # high

    # Fill NaN
    df_out['vol_regime'] = df_out['vol_regime'].fillna(1)
    df_out['trend_regime'] = df_out['trend_regime'].fillna(0)
    df_out['momentum_regime'] = df_out['momentum_regime'].fillna(1)

    return df_out


print('Advanced regime features function defined')

Feature engineering functions defined
Cross-ratio feature function defined
Advanced regime features function defined


## A2. Feature Selection Pipeline (IN-FOLD)

In [37]:
def select_features_rolling_stability(X: pd.DataFrame, y: np.ndarray, n_windows: int = 3) -> List[str]:
    """
    STEP 1: Rolling Window Stability Selection
    """
    n_samples = len(X)
    window_size = n_samples // n_windows
    
    feature_counts = {col: 0 for col in X.columns}
    
    for i in range(n_windows):
        start = i * window_size
        end = (i + 1) * window_size if i < n_windows - 1 else n_samples
        
        X_window = X.iloc[start:end]
        y_window = y[start:end]
        
        rf = RandomForestRegressor(
            n_estimators=50,
            max_depth=3,
            min_samples_leaf=30,
            random_state=42 + i,
            n_jobs=-1
        )
        rf.fit(X_window, y_window)
        
        importances = pd.Series(rf.feature_importances_, index=X.columns)
        threshold = importances.quantile(0.70)
        
        for col in X.columns:
            if importances[col] >= threshold:
                feature_counts[col] += 1
    
    selected = [col for col, count in feature_counts.items() if count >= 2]
    return selected
def select_features_hierarchical_clustering(X: pd.DataFrame, y: np.ndarray, n_clusters: int = 5) -> List[str]:
    """
    STEP 2: Hierarchical Clustering
    """
    X = X.copy().fillna(0)
    
    corr_matrix = X.corr(method='spearman')
    corr_matrix = corr_matrix.fillna(0.0).clip(-1.0, 1.0)
    
    distance_matrix = np.sqrt(0.5 * (1 - corr_matrix))
    
    clustering = AgglomerativeClustering(n_clusters=n_clusters)
    cluster_labels = clustering.fit_predict(distance_matrix)
    
    rf = RandomForestRegressor(
        n_estimators=50,
        max_depth=3,
        min_samples_leaf=30,
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X, y)
    feature_importance = pd.Series(rf.feature_importances_, index=X.columns)
    
    selected = []
    for cluster_id in range(n_clusters):
        cluster_features = X.columns[cluster_labels == cluster_id].tolist()
        if cluster_features:
            best_feature = feature_importance[cluster_features].idxmax()
            selected.append(best_feature)
    
    return selected
def create_minimal_interactions(X: pd.DataFrame, core_features: List[str], max_interactions: int = 10) -> pd.DataFrame:
    """
    STEP 3: Minimal Interactions
    """
    use_features = core_features[:min(8, len(core_features))]
    
    interactions = pd.DataFrame(index=X.index)
    count = 0
    
    for i, f1 in enumerate(use_features):
        for f2 in use_features[i+1:]:
            if count >= max_interactions:
                break
            
            interactions[f'{f1}_x_{f2}'] = X[f1] * X[f2]
            count += 1
            
            if count >= max_interactions:
                break
            
            interactions[f'{f1}_div_{f2}'] = X[f1] / (np.abs(X[f2]) + 1e-8)
            count += 1
            
            if count >= max_interactions:
                break
        
        if count >= max_interactions:
            break
    
    interactions = interactions.replace([np.inf, -np.inf], np.nan).fillna(0)
    return interactions
print('✅ Feature selection functions defined')


✅ Feature selection functions defined


## A3. Time-Series CV

In [38]:
def create_expanding_window_cv(n_samples: int, n_folds: int = 5, min_train_size: int = 2000) -> List[Tuple[np.ndarray, np.ndarray]]:
    """
    Create expanding window time-series CV splits.
    """
    remaining = n_samples - min_train_size
    fold_size = remaining // n_folds
    
    splits = []
    for i in range(n_folds):
        train_end = min_train_size + i * fold_size
        valid_start = train_end
        valid_end = min(train_end + fold_size, n_samples)
        
        train_idx = np.arange(0, train_end)
        valid_idx = np.arange(valid_start, valid_end)
        
        splits.append((train_idx, valid_idx))
    
    return splits
print('✅ Time-series CV defined')


✅ Time-series CV defined


## A4. GridSearch + Stacking Model Training

In [39]:
# GridSearch Configuration - PHASE 1: Enhanced with regularization
PARAM_GRID = {
    'max_depth': [4, 5, 6],
    'learning_rate': [0.02, 0.03, 0.04],
    'n_estimators': [300, 400, 500],
    'min_child_samples': [40, 50, 60],
    'feature_fraction': [0.7, 0.8, 0.9],
    'bagging_fraction': [0.7, 0.8],
}

param_combinations = list(product(
    PARAM_GRID['max_depth'],
    PARAM_GRID['learning_rate'],
    PARAM_GRID['n_estimators'],
    PARAM_GRID['min_child_samples'],
    PARAM_GRID['feature_fraction'],
    PARAM_GRID['bagging_fraction']
))

print(f'GridSearch: {len(param_combinations)} combinations per fold')


def gridsearch_lgbm(X_train, y_train, X_valid, y_valid, param_combinations, verbose=True):
    """
    Comprehensive GridSearch for LightGBM.
    """
    best_score = -np.inf
    best_params = None
    best_model = None

    for idx, (max_depth, learning_rate, n_estimators, min_child_samples, feature_fraction, bagging_fraction) in enumerate(param_combinations):
        params = {
            'objective': 'regression',
            'metric': 'rmse',
            'boosting_type': 'gbdt',
            'max_depth': max_depth,
            'num_leaves': min(2**max_depth - 1, 31),
            'learning_rate': learning_rate,
            'n_estimators': n_estimators,
            'subsample': bagging_fraction,
            'colsample_bytree': feature_fraction,
            'min_child_samples': min_child_samples,
            'reg_alpha': 0.1,
            'reg_lambda': 1.0,
            'verbose': -1,
            'random_state': 42
        }

        model = lgb.LGBMRegressor(**params)
        model.fit(X_train, y_train)

        pred = model.predict(X_valid)
        score = np.corrcoef(pred, y_valid)[0, 1]

        if score > best_score:
            best_score = score
            best_params = params.copy()
            best_model = model

    if verbose:
        print(f'  Best: depth={best_params["max_depth"]}, lr={best_params["learning_rate"]:.3f}, n_est={best_params["n_estimators"]} -> corr={best_score:.6f}')

    return best_params, best_model


def train_diverse_base_models(X_train, y_train, X_valid, y_valid, param_combinations):
    """
    PHASE 3: Train diverse base models (not just LightGBM variants).
    Returns ensemble prediction from multiple model types.
    """
    print('  [PHASE 3] Training diverse base models ensemble...')

    models = []
    predictions = []

    # Model 1: Standard LightGBM (with GridSearch)
    print('    [1/4] LightGBM standard (GridSearch)...')
    best_params, best_model = gridsearch_lgbm(X_train, y_train, X_valid, y_valid, param_combinations, verbose=False)
    pred1 = best_model.predict(X_valid)
    models.append(best_model)
    predictions.append(pred1)
    corr1 = np.corrcoef(pred1, y_valid)[0, 1]
    print(f'      Correlation: {corr1:.6f}')

    # Model 2: LightGBM with DART boosting (dropout regularization)
    print('    [2/4] LightGBM DART...')
    dart_params = best_params.copy()
    dart_params['boosting_type'] = 'dart'
    dart_params['drop_rate'] = 0.1
    dart_params['n_estimators'] = 400
    dart_model = lgb.LGBMRegressor(**dart_params)
    dart_model.fit(X_train, y_train)
    pred2 = dart_model.predict(X_valid)
    models.append(dart_model)
    predictions.append(pred2)
    corr2 = np.corrcoef(pred2, y_valid)[0, 1]
    print(f'      Correlation: {corr2:.6f}')

    # Model 3: Ridge Regression (linear baseline)
    print('    [3/4] Ridge Regression...')
    from sklearn.linear_model import Ridge
    ridge = Ridge(alpha=1.0)
    ridge.fit(X_train, y_train)
    pred3 = ridge.predict(X_valid)
    models.append(ridge)
    predictions.append(pred3)
    corr3 = np.corrcoef(pred3, y_valid)[0, 1]
    print(f'      Correlation: {corr3:.6f}')

    # Model 4: Heavily regularized LightGBM
    print('    [4/4] LightGBM (heavy regularization)...')
    reg_params = {
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'max_depth': 3,
        'num_leaves': 7,
        'learning_rate': 0.05,
        'n_estimators': 300,
        'subsample': 0.6,
        'colsample_bytree': 0.6,
        'min_child_samples': 70,
        'reg_alpha': 2.0,
        'reg_lambda': 5.0,
        'verbose': -1,
        'random_state': 42
    }
    reg_model = lgb.LGBMRegressor(**reg_params)
    reg_model.fit(X_train, y_train)
    pred4 = reg_model.predict(X_valid)
    models.append(reg_model)
    predictions.append(pred4)
    corr4 = np.corrcoef(pred4, y_valid)[0, 1]
    print(f'      Correlation: {corr4:.6f}')

    # Weighted ensemble (inverse variance weighting)
    variances = [np.var(p) for p in predictions]
    weights = [1/v if v > 0 else 1.0 for v in variances]
    weights = np.array(weights) / sum(weights)

    ensemble_pred = sum(w * p for w, p in zip(weights, predictions))
    ensemble_corr = np.corrcoef(ensemble_pred, y_valid)[0, 1]

    print(f'    Ensemble correlation: {ensemble_corr:.6f}')
    print(f'    Weights: {[f"{w:.3f}" for w in weights]}')

    return ensemble_pred, models, best_params


def train_meta_model_enhanced(base_pred_train, X_train_core, y_train, base_pred_valid, X_valid_core, y_valid):
    """
    PHASE 2: Enhanced meta-model with GridSearch and richer features.

    FIXED: Now properly receives y_valid to score validation predictions
    """
    # Build meta features
    X_meta_train = pd.DataFrame({
        'base_pred': base_pred_train,
        'base_pred_sq': base_pred_train ** 2,
        'base_pred_abs': np.abs(base_pred_train),
    })

    # Add top core features (limit to 10 to avoid overfitting)
    n_core_features = min(10, len(X_train_core.columns))
    for col in X_train_core.columns[:n_core_features]:
        X_meta_train[col] = X_train_core[col].values

    # GridSearch on meta-model
    meta_grid = {
        'max_depth': [2, 3, 4],
        'learning_rate': [0.03, 0.05, 0.07],
        'n_estimators': [100, 150, 200],
    }

    best_meta_score = -np.inf
    best_meta_model = None

    for depth in meta_grid['max_depth']:
        for lr in meta_grid['learning_rate']:
            for n_est in meta_grid['n_estimators']:
                params = {
                    'objective': 'regression',
                    'metric': 'rmse',
                    'max_depth': depth,
                    'num_leaves': min(2**depth, 15),
                    'learning_rate': lr,
                    'n_estimators': n_est,
                    'subsample': 0.8,
                    'colsample_bytree': 0.8,
                    'min_child_samples': 20,
                    'verbose': -1,
                    'random_state': 42
                }

                model = lgb.LGBMRegressor(**params)
                model.fit(X_meta_train, y_train)

                # Build validation meta features
                X_meta_valid_temp = pd.DataFrame({
                    'base_pred': base_pred_valid,
                    'base_pred_sq': base_pred_valid ** 2,
                    'base_pred_abs': np.abs(base_pred_valid),
                })
                for col in X_valid_core.columns[:n_core_features]:
                    X_meta_valid_temp[col] = X_valid_core[col].values

                pred = model.predict(X_meta_valid_temp)
                # FIX: Use y_valid instead of y_train for validation scoring
                score = np.corrcoef(pred, y_valid)[0, 1]

                if score > best_meta_score:
                    best_meta_score = score
                    best_meta_model = model

    # Final prediction with best model
    X_meta_valid = pd.DataFrame({
        'base_pred': base_pred_valid,
        'base_pred_sq': base_pred_valid ** 2,
        'base_pred_abs': np.abs(base_pred_valid),
    })
    for col in X_valid_core.columns[:n_core_features]:
        X_meta_valid[col] = X_valid_core[col].values

    meta_pred = best_meta_model.predict(X_meta_valid)
    return meta_pred, best_meta_model


print('Model training functions defined')


GridSearch: 486 combinations per fold
Model training functions defined


In [40]:
# Helper functions for temporal smoothing and position conversion

def temporal_smooth_predictions(predictions: np.ndarray, window: int = 5) -> np.ndarray:
    """
    PHASE 3: Apply temporal smoothing to predictions using median filter.
    Reduces noise and improves stability.
    """
    from scipy.signal import medfilt

    # Apply median filter
    smoothed = medfilt(predictions, kernel_size=window)

    # Blend: 70% original + 30% smoothed
    alpha = 0.7
    blended = alpha * predictions + (1 - alpha) * smoothed

    return blended


def returns_to_position(return_preds, multiplier):
    """
    Convert return predictions to positions.

    Formula: pos = 1 + pred * multiplier
    Clipped to [0, 2]
    """
    pos = 1.0 + return_preds * multiplier
    return np.clip(pos, 0.0, 2.0)


print('Helper functions defined')


Helper functions defined


## A5. Execute Complex Pipeline (Generate OOF)

In [41]:
print('\n' + '='*70)
print('PART A: COMPLEX MODEL TRAINING (OOF GENERATION)')
print('='*70)

# Step 1: Feature Engineering
print('\n[A1] Creating features...')
train['forward_returns'] = train['market_forward_excess_returns'] + train['risk_free_rate']

M_cols = [c for c in raw_features if c.startswith('M')]
V_cols = [c for c in raw_features if c.startswith('V')]
D_cols = [c for c in raw_features if c.startswith('D')]
E_cols = [c for c in raw_features if c.startswith('E')]
key_cols = M_cols + V_cols + D_cols + E_cols

train_fe = create_safe_rolling_features(train, key_cols, windows=[5, 10, 21])
train_fe = create_safe_regime_features(train_fe)
train_fe = create_cross_ratio_features(train_fe, raw_features)  # PHASE 2
train_fe = create_advanced_regime_features(train_fe)  # PHASE 3

engineered_cols = [c for c in train_fe.columns if c not in train.columns]
all_feature_cols = raw_features + engineered_cols

X = train_fe[all_feature_cols].copy()
y = train_fe[target_col].copy()

X = X.fillna(method="ffill").fillna(0)
y = y.fillna(0)

print(f'  Total features: {len(all_feature_cols)} ({len(raw_features)} raw + {len(engineered_cols)} engineered)')
print(f'  Final X shape: {X.shape}')

# Step 2: CV Setup
print('\n[A2] Setting up CV...')
n_folds = 5
min_train_size = 2000
cv_splits = create_expanding_window_cv(len(X), n_folds=n_folds, min_train_size=min_train_size)

for i, (train_idx, valid_idx) in enumerate(cv_splits):
    print(f'  Fold {i+1}: train={len(train_idx):4d}, valid={len(valid_idx):3d}')

# Step 3: Train Complex Model with IN-FOLD Feature Selection
print('\n[A3] Training complex model (GridSearch + Stacking)...')

oof_base_predictions = np.zeros(len(X))
oof_meta_predictions = np.zeros(len(X))
all_base_models = []
all_meta_models = []

start_time = time.time()

for fold_idx, (train_idx, valid_idx) in enumerate(cv_splits):
    fold_start = time.time()

    print(f'\n  Fold {fold_idx + 1}/{n_folds}')

    X_train_raw = X.iloc[train_idx].copy()
    X_valid_raw = X.iloc[valid_idx].copy()
    y_train = y.iloc[train_idx].values
    y_valid = y.iloc[valid_idx].values

    # IN-FOLD Feature Selection
    stable_features = select_features_rolling_stability(X_train_raw, y_train, n_windows=3)
    X_train_stable = X_train_raw[stable_features]
    X_valid_stable = X_valid_raw[stable_features]

    cluster_features = select_features_hierarchical_clustering(X_train_stable, y_train, n_clusters=12)  # PHASE 1: Increased from 5
    X_train_core = X_train_stable[cluster_features]
    X_valid_core = X_valid_stable[cluster_features]

    interactions_train = create_minimal_interactions(X_train_stable, cluster_features, max_interactions=25)
    interactions_valid = create_minimal_interactions(X_valid_stable, cluster_features, max_interactions=25)

    X_train_fold = pd.concat([X_train_core, interactions_train], axis=1)
    X_valid_fold = pd.concat([X_valid_core, interactions_valid], axis=1)

    # Scaling
    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(
        scaler.fit_transform(X_train_fold),
        columns=X_train_fold.columns,
        index=X_train_fold.index
    )
    X_valid_scaled = pd.DataFrame(
        scaler.transform(X_valid_fold),
        columns=X_valid_fold.columns,
        index=X_valid_fold.index
    )

    # Base Models with GridSearch
    base_pred_valid, base_models, best_params = train_diverse_base_models(  # PHASE 3: Diverse ensemble
        X_train_scaled, y_train,
        X_valid_scaled, y_valid,
        param_combinations
    )
    base_pred_train = np.mean([m.predict(X_train_scaled) for m in base_models], axis=0)

    # Meta Model - FIX: Pass y_valid as argument
    print('  Training meta model...')
    meta_pred_valid, meta_model = train_meta_model_enhanced(  # PHASE 2: Enhanced meta
        base_pred_train,
        X_train_scaled[cluster_features],
        y_train,
        base_pred_valid,
        X_valid_scaled[cluster_features],
        y_valid  # FIX: Added y_valid parameter
    )

    # Store OOF
    oof_base_predictions[valid_idx] = base_pred_valid
    oof_meta_predictions[valid_idx] = meta_pred_valid
    all_base_models.append((base_models, scaler, cluster_features))
    all_meta_models.append(meta_model)

    fold_corr = np.corrcoef(meta_pred_valid, y_valid)[0, 1]
    fold_elapsed = time.time() - fold_start

    print(f'  Correlation: {fold_corr:.6f} | Time: {fold_elapsed/60:.1f} min')

total_elapsed = time.time() - start_time

# PHASE 3: Apply temporal smoothing to OOF predictions
oof_meta_predictions_raw = oof_meta_predictions.copy()
oof_meta_predictions = temporal_smooth_predictions(oof_meta_predictions, window=5)

oof_corr_raw = np.corrcoef(oof_meta_predictions_raw, y.values)[0, 1]
oof_corr = np.corrcoef(oof_meta_predictions, y.values)[0, 1]

print('\n' + '='*70)
print('COMPLEX MODEL COMPLETE')
print('='*70)
print(f'Final OOF Correlation (raw): {oof_corr_raw:.6f}')
print(f'Final OOF Correlation (smoothed): {oof_corr:.6f}')
print(f'Total training time: {total_elapsed/60:.1f} minutes')
print('='*70)



PART A: COMPLEX MODEL TRAINING (OOF GENERATION)

[A1] Creating features...
  Total features: 808 (94 raw + 714 engineered)
  Final X shape: (9048, 808)

[A2] Setting up CV...
  Fold 1: train=2000, valid=1409
  Fold 2: train=3409, valid=1409
  Fold 3: train=4818, valid=1409
  Fold 4: train=6227, valid=1409
  Fold 5: train=7636, valid=1409

[A3] Training complex model (GridSearch + Stacking)...

  Fold 1/5
  [PHASE 3] Training diverse base models ensemble...
    [1/4] LightGBM standard (GridSearch)...
      Correlation: 0.110397
    [2/4] LightGBM DART...
      Correlation: 0.094492
    [3/4] Ridge Regression...
      Correlation: 0.067182
    [4/4] LightGBM (heavy regularization)...
      Correlation: 0.000000
    Ensemble correlation: 0.000000
    Weights: ['0.000', '0.000', '0.000', '1.000']
  Training meta model...
  Correlation: -0.033242 | Time: 0.8 min

  Fold 2/5
  [PHASE 3] Training diverse base models ensemble...
    [1/4] LightGBM standard (GridSearch)...
      Correlation: -

---
# PARTIE B: MULTIPLIER OPTIMIZATION (Using Complex OOF)
---

In [42]:
print('\n' + '='*70)
print('PART B: MULTIPLIER OPTIMIZATION (FROM COMPLEX OOF)')
print('='*70)

# Get actual returns and risk-free rate
forward_returns = train['forward_returns'].values
risk_free = train['risk_free_rate'].values

# Search for best multiplier
# PHASE 2: Two-stage multiplier optimization
print('\nPHASE 2: Two-stage multiplier optimization')

print('  Stage 1: Coarse search...')
coarse_grid = np.linspace(50, 600, 50)
best_m_coarse = None
best_score_coarse = -999

for m in coarse_grid:
    pos = returns_to_position(oof_meta_predictions, m)
    strat_ret = risk_free * (1 - pos) + pos * forward_returns
    score = competition_score(strat_ret, risk_free, forward_returns)

    if score > best_score_coarse:
        best_score_coarse = score
        best_m_coarse = m

print(f'  Coarse: m={best_m_coarse:.1f}, score={best_score_coarse:.5f}')

print('  Stage 2: Fine search...')
fine_grid = np.linspace(max(50, best_m_coarse-50), min(600, best_m_coarse+50), 200)
best_multiplier = None
best_score = -999

for m in fine_grid:
    pos = returns_to_position(oof_meta_predictions, m)
    strat_ret = risk_free * (1 - pos) + pos * forward_returns
    score = competition_score(strat_ret, risk_free, forward_returns)

    if score > best_score:
        best_score = score
        best_multiplier = m

print(f'  Fine: m={best_multiplier:.2f}, score={best_score:.5f}')

print('\nOPTIMAL MULTIPLIER FOUND')
print('='*70)
print(f'Best multiplier: {best_multiplier:.2f}')
print(f'Competition score (complex OOF): {best_score:.5f}')
print('='*70)

# Store for later use
OPTIMIZED_MULTIPLIER = best_multiplier



PART B: MULTIPLIER OPTIMIZATION (FROM COMPLEX OOF)

PHASE 2: Two-stage multiplier optimization
  Stage 1: Coarse search...
  Coarse: m=195.9, score=-0.04701
  Stage 2: Fine search...
  Fine: m=202.20, score=-0.04698

OPTIMAL MULTIPLIER FOUND
Best multiplier: 202.20
Competition score (complex OOF): -0.04698


---
# PARTIE C: SIMPLE INFERENCE MODEL (Raw Features Only)
---

In [43]:
print('\n' + '='*70)
print('PART C: SIMPLE INFERENCE MODEL (RAW FEATURES ONLY)')
print('='*70)
# Prepare raw features for inference
inference_features = [c for c in train.columns if c.startswith(("D","E","I","M","P","S","V"))]
print(f'\nUsing {len(inference_features)} raw features for inference model.')
X_inf = train[inference_features].fillna(0.0)
y_inf = train[target_col].fillna(0.0).values
# Scale
inf_scaler = StandardScaler()
X_inf_scaled = inf_scaler.fit_transform(X_inf)
# Train simple LightGBM model
print('\nTraining simple LightGBM model...')
inf_model = lgb.LGBMRegressor(
    max_depth=4,
    learning_rate=0.03,
    n_estimators=400,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    verbose=-1
)
inf_model.fit(X_inf_scaled, y_inf)
# Store artifacts
model_artifacts = {
    "features": inference_features,
    "scaler": inf_scaler,
    "model": inf_model,
    "multiplier": OPTIMIZED_MULTIPLIER,
    "MIN_INVESTMENT": 0.0,
    "MAX_INVESTMENT": 2.0,
}
print(f'\n✅ SIMPLE INFERENCE MODEL READY')
print('='*70)
print(f'Features: {len(inference_features)} raw features')
print(f'Multiplier: {OPTIMIZED_MULTIPLIER:.2f} (from complex OOF)')
print('='*70)



PART C: SIMPLE INFERENCE MODEL (RAW FEATURES ONLY)

Using 94 raw features for inference model.

Training simple LightGBM model...

✅ SIMPLE INFERENCE MODEL READY
Features: 94 raw features
Multiplier: 202.20 (from complex OOF)


---
# PARTIE D: KAGGLE predict() FUNCTION
---

In [44]:
print('\n' + '='*70)
print('PART D: KAGGLE predict() FUNCTION')
print('='*70)
try:
    import polars as pl
    print('✅ Polars imported successfully')
except ImportError:
    print('⚠ Polars not available (will be available in Kaggle)')
def predict(test: "pl.DataFrame") -> float:
    """
    Kaggle inference server predict function.
    
    Args:
        test: Polars DataFrame with one row of test features.
    
    Returns:
        float: Position allocation between 0.0 and 2.0.
    """
    try:
        # Convert to pandas
        test_pd = test.to_pandas()
        
        # Extract raw features (missing → 0)
        X_row = np.zeros((1, len(model_artifacts["features"])))
        for i, feat in enumerate(model_artifacts["features"]):
            if feat in test_pd.columns:
                val = test_pd[feat].fillna(0).values[0]
                X_row[0, i] = val
        
        # Scale
        X_row_scaled = model_artifacts["scaler"].transform(X_row)
        
        # Predict return
        ret_pred = model_artifacts["model"].predict(X_row_scaled)[0]
        
        # Convert to position using optimized multiplier
        mult = model_artifacts["multiplier"]
        pos = 1.0 + ret_pred * mult
        pos = np.clip(pos, model_artifacts["MIN_INVESTMENT"], model_artifacts["MAX_INVESTMENT"])
        
        return float(pos)
    
    except Exception as e:
        # Fallback: neutral position
        print(f"Error in predict(): {e}")
        return 1.0
print('\n✅ predict() function READY for Kaggle submission')
print(f'  - Uses {len(model_artifacts["features"])} raw features')
print(f'  - Optimized multiplier: {model_artifacts["multiplier"]:.2f}')
print(f'  - Output range: [0.0, 2.0]')



PART D: KAGGLE predict() FUNCTION
⚠ Polars not available (will be available in Kaggle)

✅ predict() function READY for Kaggle submission
  - Uses 94 raw features
  - Optimized multiplier: 202.20
  - Output range: [0.0, 2.0]


## D2. Test predict() Function Locally

In [45]:
print('\n' + '='*70)
print('LOCAL TEST OF predict() FUNCTION')
print('='*70)
try:
    import polars as pl
    test_pl = pl.from_pandas(test.copy())
    
    preds = []
    for i in range(len(test_pl)):
        row = test_pl[i:i+1]
        pos = predict(row)
        preds.append(pos)
        if i < 5:
            date_id = row.select("date_id").to_series().item()
            print(f"  date_id {date_id}: position = {pos:.4f}")
    
    preds = np.array(preds)
    print(f"\nPrediction Statistics (test):")
    print(f"  Mean: {preds.mean():.3f}")
    print(f"  Std: {preds.std():.3f}")
    print(f"  Range: [{preds.min():.3f}, {preds.max():.3f}]")
    print(f"  All within [0, 2]: {((preds >= 0) & (preds <= 2)).all()}")
except Exception as e:
    print(f"Local test skipped: {e}")



LOCAL TEST OF predict() FUNCTION
Local test skipped: No module named 'polars'


---
# PARTIE E: KAGGLE INFERENCE SERVER (Optional)
---

In [46]:
print('\n' + '='*70)
print('KAGGLE INFERENCE SERVER')
print('='*70)
try:
    import os
    import kaggle_evaluation.default_inference_server as deis
    
    inference_server = deis.DefaultInferenceServer(predict)
    
    if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
        print("\n>>> Running in Kaggle competition mode - serving predictions...")
        inference_server.serve()
    else:
        print("\n>>> Running locally - testing with gateway...")
        inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))
except Exception as e:
    print(f"⚠ Could not start inference server: {e}")
    print("This is normal outside the Kaggle environment.")



KAGGLE INFERENCE SERVER
⚠ Could not start inference server: No module named 'polars'
This is normal outside the Kaggle environment.
